# Single-cell RNA-seq preprocessing and annotation

Analysis notebook for **Oncolytic membrane-bound CD40L rewires macrophages to boost CD8+ T cell responses and antitumor immunity**.

The input location and output root are configured in the first Python section. The DOI shown in the original template was unrelated to this study and has been removed.


In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt
import scvelo as scv
import pandas as pd
import numpy as np
import os
from pathlib import Path
import scanpy as sc
import scipy.stats as stats
from io import StringIO
from sklearn import linear_model
from sklearn.svm import SVR
import seaborn as sns
import scanpy.external as sce
import scrublet as scr
%matplotlib inline


In [ ]:
import scvi
import torch
%matplotlib inline


In [ ]:
scvi.settings.seed = 0
plt.rcParams["pdf.fonttype"]=42
plt.rcParams["ps.fonttype"]=42

In [ ]:
sc.set_figure_params(figsize=(4, 4))
torch.set_float32_matmul_precision("high")


In [ ]:
# Project configuration. Environment variables allow the notebook to run
# without editing code on another machine.
project_ID = "pool"
mtx = os.getenv("OAD_DATA_DIR", "transcriptome")

configured_input = os.getenv("OAD_POOL_H5AD")
input_candidates = [Path(mtx) / "pool_raw.h5ad", Path("pool_raw.h5ad")]
input_h5ad = Path(configured_input) if configured_input else next(
    (path for path in input_candidates if path.exists()), input_candidates[0]
)
if not input_h5ad.exists():
    raise FileNotFoundError(
        f"Pooled AnnData file not found: {input_h5ad}. "
        "Set OAD_POOL_H5AD or place pool_raw.h5ad under OAD_DATA_DIR."
    )

sc.settings.figdir = f"{mtx}/{project_ID}_figures"
scv.settings.figdir = sc.settings.figdir
os.makedirs(sc.settings.figdir, exist_ok=True)
os.makedirs(f"{mtx}/{project_ID}_DEG", exist_ok=True)


In [ ]:
print(f"Input: {input_h5ad.resolve()}")
print(f"Output root: {Path(mtx).resolve()}")


In [ ]:
adata = sc.read(input_h5ad)
adata.var_names_make_unique("_")


In [ ]:
# Output directories were created in the configuration cell.
print(f"Loaded {adata.n_obs:,} cells and {adata.n_vars:,} genes.")


## Preprocess the Data

now we will take the files from the previous section where we determined spliced/unspliced RNA counts and use them to create RNA velocity trajectory as well as investigate biological alternation in each cell

In [ ]:
# Prepare the pooled full-gene expression reference before HVG selection.
adata_raw = adata.copy()


In [ ]:
sc.pp.filter_genes(adata_raw, min_counts=3)
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)


In [ ]:
# Store the pooled normalized, log1p expression reference once.
adata.raw = adata_raw


**Continued data processing**

In [ ]:
adata.layers["counts"] = adata.X.copy()  # preserve counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="Sample_id",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    categorical_covariate_keys=["Sample_id"],
    continuous_covariate_keys=['n_genes_by_counts','pct_counts_mt','total_counts', 'pct_counts_Rp'],
)

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model_dir = os.path.join(mtx, f"{project_ID}_figures", "scvi_model")
model.save(model_dir, overwrite=True)


In [ ]:
model = scvi.model.SCVI.load(model_dir, adata=adata)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
denoised = model.get_normalized_expression(adata, library_size=1e4)

In [ ]:
SCVI_NORMALIZED_KEY = "scvi_normalized"

adata.layers[SCVI_NORMALIZED_KEY] = model.get_normalized_expression(library_size=10e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
# neighbors were already computed using scVI
SCVI_CLUSTERS_KEY = "leiden_scVI"
sc.tl.leiden(adata, key_added=SCVI_CLUSTERS_KEY, resolution=0.4)

In [ ]:
old_to_new={
    "0":"T.cells",
    "1":"ID8.cells",
    "2":"Myeloid.cells",
    "3":"T.cells",
    "4":"B.cells",
    "5":"Myeloid.cells",
    "6":"B.cells",
    "7":"T.cells",
    "8":"Epi.cells",
    "9":"NK.cells",
    "10":"Unknown.cells",
    "11":"Endo.cells",
    "12":"Myeloid.cells",
   # "13":"mixed.cells",
    #"14":"T.cells",
    #"15":"Myeloid.cells",
    #"16":"B.cells",
    #"17":"Epi.cells",
    #"18":"Fibro.cells",
    #"19":"T.cells",
    #"20":"Epi.cells",
    #"21":"Epi.cells",
    #"22":"Myeloid.cells",
    #"23":"T.cells",
    #"24":"T.cells",
    #"25":"Myeloid.cells",
   # "26":"Epi.cells"
}

In [ ]:
adata.obs["anno_clusters"]=(adata.obs["leiden_scVI"].map(old_to_new).astype("category"))

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_clusters"],
    frameon=False,
)
sc.pl.umap(
    adata,
    color=["Sample_id","treatment"],
    ncols=2,
    frameon=False,
)

In [ ]:
sc.tl.dendrogram(adata,groupby="anno_clusters")
markers ={
    "T.cells":["Ptprc","Cd2","Cd3e","Cd3d","Cd4","Tcf7","Foxp3","Ctla4","Cd8a","Ccr7","Pdcd1","Trdc","Trgc1","Mki67","Ncr1","Nkg7","Prf1","Gzma","Gzmb","Ifng"],
    "B.cells":["Cd79a","Cd19","Ms4a1","Jchain","Ighd"],
    "Dendritic.cells":["Clec10a","Itgax","Cd300c"],
    "Neutrophil.cells":["Fcgr3"],
    "Myeloid.cells":["Itgam","Cd14","Adgre1","Cd33","Cd68","Csf1r","Fcgr2b","Tnf", "Il1a","Cd86",
                     "Cd44","Mrc1","Siglec1","Arg1","Ccl8","C5ar1"],
     "Epithelial.cells":["Epcam","Cdh1","Krt8","Krt18"],   
    "Fibroblasts":["Dcn","Pdpn","Acta2",
"Tagln","Myl9","Tpm2","Cxcl1","Il6","Has1"],
    "Endothelial.cells":["Epas1","Sparcl1"],
   
    }

sc.pl.dotplot(adata, markers,standard_scale="var",cmap="Spectral_r"
              ,#figsize=(12,4),
                     mean_only_expressed=False, expression_cutoff=0,
                     groupby='anno_clusters', dendrogram=False,save=f"{project_ID}_markers.pdf")

In [ ]:
adata.obs["Treatment_Sample"]=adata.obs["treatment"].astype(str)+"_"+adata.obs["Sample_id"].astype(str)
adata.obs["Treatment_Sample"]=adata.obs["Treatment_Sample"].astype("category")

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi.h5ad")

In [ ]:
# Export original counts in X and inherit the pooled normalized/log1p raw reference.
os.makedirs(mtx, exist_ok=True)
major_group_labels = {
    "B": "B.cells", "T": "T.cells", "Myeloid": "Myeloid.cells",
    "ID8": "ID8.cells", "Epi": "Epi.cells", "Fibro": "Fibro.cells",
    "Endo": "Endo.cells", "NK": "NK.cells", "Unknown": "Unknown.cells",
}
exported_major_groups = set()
pooled_raw = sc.read(input_h5ad)
pooled_raw.var_names_make_unique("_")
for group_name, group_label in major_group_labels.items():
    group_cells = adata.obs_names[adata.obs["anno_clusters"].eq(group_label).fillna(False)]
    if len(group_cells) == 0:
        print(f"{group_name}: no cells with broad label {group_label}; skipped export.")
        continue
    group_raw = pooled_raw[group_cells, :].copy()
    group_raw.obs = adata.obs.loc[group_cells].copy()
    group_raw.raw = adata[group_cells, :].raw.to_adata()
    group_raw.write(f"{mtx}/pool_{group_name}_raw.h5ad")
    exported_major_groups.add(group_name)
del pooled_raw


In [ ]:
#set the directory for outputs
project_ID="pool_B"
mtx="transcriptome"
sc.settings.figdir=f'{mtx}/{project_ID}_figures'
sc.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
scv.settings.seed = 0
plt.rcParams["pdf.fonttype"]=42
plt.rcParams["ps.fonttype"]=42

In [ ]:
scv.settings.figdir=f'{mtx}/{project_ID}_figures'
scv.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
sc.set_figure_params(figsize=(4,4))
torch.set_float32_matmul_precision("high")

In [ ]:
if "B" not in exported_major_groups:
    raise ValueError("B: no cells exported from current broad annotation; check anno_clusters.")
adata=sc.read(f"{mtx}/{project_ID}_raw.h5ad")

In [ ]:
#store a h5ad file for further faster loadinig

#we need to make a directory to store the files in
try:
    directoryName=mtx+"/"+project_ID+"_figures"
    os.mkdir(directoryName)
except:
    pass

#we need to make a directory to store the files in

try:
    directoryName=mtx+"/"+project_ID+"_DEG"
    os.mkdir(directoryName)
except:
    pass


## Preprocess the Data

now we will take the files from the previous section where we determined spliced/unspliced RNA counts and use them to create RNA velocity trajectory as well as investigate biological alternation in each cell

In [ ]:
#plot scatter plots of dataset QC
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt',color="anno_clusters")
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_Rp',color="anno_clusters")
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts',color="anno_clusters")

In [ ]:

adata = adata[adata.obs.n_genes_by_counts < 7000, :] 
#adata = adata[adata.obs.n_genes_by_counts > 500, :] 
adata = adata[adata.obs.total_counts < 40000, :]
#adata = adata[adata.obs.total_counts > 1000, :]
#adata = adata[adata.obs.pct_counts_mt < 5, :]

**Continued data processing**

In [ ]:
adata.layers["counts"] = adata.X.copy()  # preserve counts

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="Sample_id",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    categorical_covariate_keys=["Sample_id"],
    continuous_covariate_keys=['n_genes_by_counts','pct_counts_mt','total_counts', 'pct_counts_Rp'],
)

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model_dir = os.path.join(mtx, f"{project_ID}_figures", "scvi_model")
model.save(model_dir, overwrite=True)


In [ ]:
model = scvi.model.SCVI.load(model_dir, adata=adata)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
denoised = model.get_normalized_expression(adata, library_size=1e4)

In [ ]:
SCVI_NORMALIZED_KEY = "scvi_normalized"

adata.layers[SCVI_NORMALIZED_KEY] = model.get_normalized_expression(library_size=10e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)

In [ ]:
sc.pp.scale(adata,layer=SCVI_NORMALIZED_KEY )

In [ ]:
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_clusters"],
    frameon=False,
)
sc.pl.umap(
    adata,
    color=["Sample_id","treatment"],
    ncols=2,
    frameon=False,
)

In [ ]:
# neighbors were already computed using scVI
SCVI_CLUSTERS_KEY = "leiden_scVI"
sc.tl.leiden(adata, key_added=SCVI_CLUSTERS_KEY, resolution=0.37)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_scVI"],
    frameon=False,
)

In [ ]:
groupby="leiden_scVI"
sc.tl.dendrogram(adata,groupby=groupby)
markers ={
    "CD8":["Cd3g","Cd8a","Cd8b1"],
    "activated.B":[genes.capitalize() for genes in ["CD19","MS4A1","ID3","GPR183","CCR7","EMP3","BANK1","ACP5","IFITM2","CAPG","NR4A2"]],
    "antibody.secreting.B":[genes.capitalize() for genes in["CD38","MZB1","XBP1","PRDM1","SSR4","DERL3","FKBP11","ITM2C","HSP90B1","HERPUD1","SEC11C"]],
    "Gernimal.centre.B":[genes.capitalize() for genes in["CXCR5","AICDA","MME","STMN1","HMGN2","NEIL1","MARCKSL1","GMDS","BACH2","H2AZ1","TMEM123"]],
    "Transitory.cells":[genes.capitalize() for genes in ["H4C1","TYMS", "HMGB2","TUBA1B"]]
   
}


sc.pl.dotplot(adata, markers,standard_scale="var",cmap="RdYlBu_r",
                     mean_only_expressed=False, expression_cutoff=0,
                     groupby=groupby, dendrogram=True,save=f"{project_ID}_markers_{groupby}.pdf")

In [ ]:
old_to_new={
    "0":"B.Sell",
    "1":"B.secret",
    "2":"B.Bank1",
    "3":"B.APC",
    "4":"B.Lagls1",
    "5":"B.Germ",
    "6":"mixed.cells"
   
                
    
}

In [ ]:
adata.obs["anno_sub"]=(adata.obs["leiden_scVI"].map(old_to_new).astype("category"))

In [ ]:
#plot umap grouped by treatment
plot_pattern="anno_sub"
scv.pl.scatter(adata, save=f"{plot_pattern}_umap.pdf",
               basis="umap",
           color=plot_pattern,legend_loc="right",
            size=10, alpha=0.8,
            palette=[ "slategrey","PeachPuff","gold",
                     "DeepSkyBlue",
                     "SpringGreen","olivedrab",
                     "orchid","lightcoral","firebrick","crimson","Pink", "tan",
                   "MidnightBlue", "DarkMagenta", "darkviolet","royalblue", 
                   "coral","RosyBrown","Goldenrod","BurlyWood","Peru",
            "Thistle","lightseagreen","CadetBlue",
            "lightsteelblue","dimgrey","black","IndianRed","PaleVioletRed","hotpink","greenyellow","forestgreen","PaleTurquoise","Aqua",] )

# DEG

In [ ]:
sc.tl.rank_genes_groups(adata, "anno_sub", method='wilcoxon', min_log_fold_change=2, 
                        n_genes=4000,#layer="scvi_normalized",
                        use_raw=True) #when we set layer="spliced", we can extract the DEGs of each cluster according to only normalized and highly virable spliced RNA matrix. When we don't set the layer, we calculate DEGs according to scaled spliced RNA matrix. When we set use_raw="True", we will get DEGs according to raw spliced matrix even without filtering highly virable genes or outliers.
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, rankby_obs=True)

In [ ]:
#save top50 DEG into dataframe, when we calculate the DEGs according to scaled matrix, because negative numbers in the matrix, we will not be able to get log fold chages, so that we can only summarize pvals_adj in to the dataframe
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
result_df=pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names' ,'pvals_adj',"logfoldchanges"]}).head(50)

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi.h5ad")

In [ ]:
%matplotlib inline

In [ ]:
scvi.settings.seed = 0

In [ ]:
scvi.settings.seed = 0
plt.rcParams["pdf.fonttype"]=42
plt.rcParams["ps.fonttype"]=42

In [ ]:
sc.set_figure_params(figsize=(4,4))
torch.set_float32_matmul_precision("high")

In [ ]:
#set the directory for outputs
project_ID="pool_T"
mtx="transcriptome"
sc.settings.figdir=f'{mtx}/{project_ID}_figures'
sc.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
scv.settings.figdir=f'{mtx}/{project_ID}_figures'
scv.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
if "T" not in exported_major_groups:
    raise ValueError("T: no cells exported from current broad annotation; check anno_clusters.")
adata=sc.read(f"{mtx}/{project_ID}_raw.h5ad")

In [ ]:
#store a h5ad file for further faster loadinig

#we need to make a directory to store the files in
try:
    directoryName=mtx+"/"+project_ID+"_figures"
    os.mkdir(directoryName)
except:
    pass

#we need to make a directory to store the files in

try:
    directoryName=mtx+"/"+project_ID+"_DEG"
    os.mkdir(directoryName)
except:
    pass


## Preprocess the Data

now we will take the files from the previous section where we determined spliced/unspliced RNA counts and use them to create RNA velocity trajectory as well as investigate biological alternation in each cell

In [ ]:
#filter out outliers
adata = adata[adata.obs.n_genes_by_counts < 4000, :] #we want the cell to contain less than 5000 total gene detected in order to avoid doublets
adata = adata[adata.obs.total_counts < 13000, :]
adata = adata[adata.obs.n_genes_by_counts > 400, :] #we want the cell to contain more than 200 total gene detected in order to avoid bad cells
adata = adata[adata.obs.total_counts > 800, :]# we want the cell to contain more than 400 (200*2) total counts (UMI) detected in order to avoid bad cells
adata = adata[adata.obs.pct_counts_mt < 5, :]# we want the cell to contain less than 55 percent mitochondrial genes in order to avoid bad cells

adata

**Continued data processing**

In [ ]:
adata.layers["counts"] = adata.X.copy()  # preserve counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="Sample_id",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    categorical_covariate_keys=["Sample_id"],
    continuous_covariate_keys=['n_genes_by_counts','pct_counts_mt','total_counts', 'pct_counts_Rp'],
)

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model_dir = os.path.join(mtx, f"{project_ID}_figures", "scvi_model")
model.save(model_dir, overwrite=True)


In [ ]:
model = scvi.model.SCVI.load(model_dir, adata=adata)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
denoised = model.get_normalized_expression(adata, library_size=1e4)

In [ ]:
SCVI_NORMALIZED_KEY = "scvi_normalized"

adata.layers[SCVI_NORMALIZED_KEY] = model.get_normalized_expression(library_size=10e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_clusters"],
    frameon=False,
)
sc.pl.umap(
    adata,
    color=["Sample_id","treatment"],
    ncols=2,
    frameon=False,
)

In [ ]:
# neighbors were already computed using scVI
SCVI_CLUSTERS_KEY = "leiden_scVI"
sc.tl.leiden(adata, key_added=SCVI_CLUSTERS_KEY, resolution=0.5, use_weights=True)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_scVI"],
    frameon=False,
)

In [ ]:
old_to_new={
    "0":"CD4.naive",
    "1":"CD8.RM",
    "2":"CD8.prolif",
    "3":"CD8.cyto",
    "4":"CD8.naive",
    "5":"NK.T",
    "6":"CD4.exh",
    "7":"CD4.reg",
                
    
}

In [ ]:
adata.obs["anno_sub"]=(adata.obs["leiden_scVI"].map(old_to_new).astype("category"))

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_sub"],
    frameon=False,
)

In [ ]:
groupby="anno_sub"
#sc.tl.dendrogram(adata,groupby=groupby)
markers ={
    "T.cells":[genes.capitalize() for genes in ["Ptprc","Cd3d","Cd3e","Cd4","Cd8a","Trac","Trdc"]],
    "Cytotoxic":[genes.capitalize()  for genes in["Ncam1","Nkg7","Prf1","Cst7","Gzma","Gzmb","Gzmk","Ifng","Tnfsf10"]],
    "Exhausted":[genes.capitalize()  for genes in["Ctla4","Lag3","Pdcd1","Cd276","Tigit","Havcr2"]],
    "Regulatory":[genes.capitalize()  for genes in["Foxp3","Il2ra","Tgfb1","Tgfb3","Tgfbi","Tgfbr1"]],
    "Naive":[genes.capitalize()  for genes in["Ccr7","Lef1","Sell","Tcf7","Cd27"]],
    "Co-stimulatory":[genes.capitalize()  for genes in["Cd226","Icos","Tnfrsf14","Tnfrsf25","Tnfrsf9","Tnfrsf4","Cd27"]],
    "Proliferation":[genes.capitalize()  for genes in["Mki67","Top2a"]],
    "Apoptosis&Tolerance":[genes.capitalize()  for genes in["Gadd45b","Rnf128"]]
}


sc.pl.dotplot(adata, markers,standard_scale="var",cmap="RdYlBu_r",
                     mean_only_expressed=False, expression_cutoff=0, figsize=(10,2),
                     groupby=groupby, dendrogram=False,save=f"{project_ID}_markers_{groupby}.pdf")

In [ ]:
effector=["Tbx21","Id2","Prdm1","Stat4","Mafa"]
Memory=["Id3","Bcl6","Tcf7","Stat3","Foxo1","Eomes","Tnfrsf4"]
Memory_lo=["Nfatc1","Nfatc2","Nfatc3","Nfatc4","Jun","Prdm1","Tox"]
Resident_memory_hi=["Cd69","Itgae","Gzmk","Gzma","Gzmb"]
Resident_memory_lo=["Il2rb","Sell","Klf2","Ccr7","Zeb2"]
Migrantion=["Vcam1","Icam1"]
LongLive = ["Il7r","Tgfbr2"]
supports = ["Il7","Il15","Il21","Il2","Il10","Tgfb1"]

In [ ]:
groupby="anno_sub"
#sc.tl.dendrogram(adata,groupby=groupby)
markers ={
    #"T.cells":[genes.capitalize() for genes in ["Ptprc","Cd3d","Cd3e","Cd4","Cd8a","Trac","Trdc"]],
     "LongLive":LongLive,
       "Memory":Memory,
    "Memory_lo":Memory_lo,
    "Resident_memory_hi":Resident_memory_hi,
    "Resident_memory_lo":Resident_memory_lo,
    "Migrantion":Migrantion,
    "Cytotoxic":[genes.capitalize()  for genes in["Ncam1","Nkg7","Prf1","Cst7","Gzma","Gzmb","Gzmk","Ifng","Tnfsf10"]],
    "Exhausted":[genes.capitalize()  for genes in["Ctla4","Lag3","Pdcd1","Cd276","Tigit","Havcr2"]],
    "Regulatory":[genes.capitalize()  for genes in["Foxp3","Il2ra","Tgfb1","Tgfb3","Tgfbi","Tgfbr1"]],
    "Naive":[genes.capitalize()  for genes in["Ccr7","Lef1","Sell","Tcf7","Cd27"]],
    "Co-stimulatory":[genes.capitalize()  for genes in["Cd226","Icos","Tnfrsf14","Tnfrsf25","Tnfrsf9","Tnfrsf4","Cd44","Cd40lg"]],
    "Proliferation":[genes.capitalize()  for genes in["Mki67","Top2a"]],
    "Apoptosis&Tolerance":[genes.capitalize()  for genes in["Gadd45b","Rnf128"]],
 
   
    #"supports":supports
}


sc.pl.dotplot(adata, markers,standard_scale="var",cmap="RdYlBu_r",
                     mean_only_expressed=False, expression_cutoff=0, figsize=(15,2),
                     groupby=groupby, dendrogram=False,save=f"{project_ID}_markers_{groupby}_sub.pdf")

# DEG

In [ ]:
sc.tl.rank_genes_groups(adata, "anno_sub", method='wilcoxon', min_log_fold_change=2, 
                        n_genes=4000,#layer="scvi_normalized",
                        use_raw=True) #when we set layer="spliced", we can extract the DEGs of each cluster according to only normalized and highly virable spliced RNA matrix. When we don't set the layer, we calculate DEGs according to scaled spliced RNA matrix. When we set use_raw="True", we will get DEGs according to raw spliced matrix even without filtering highly virable genes or outliers.
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, rankby_obs=True)

In [ ]:
#save top50 DEG into dataframe, when we calculate the DEGs according to scaled matrix, because negative numbers in the matrix, we will not be able to get log fold chages, so that we can only summarize pvals_adj in to the dataframe
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
result_df=pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names' ,'pvals_adj',"logfoldchanges"]}).head(50)

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi.h5ad")

## Myeloid-cell subclustering


In [ ]:
#import scvi 
%matplotlib inline

In [ ]:
#scvi.settings.seed = 0
plt.rcParams["pdf.fonttype"]=42
plt.rcParams["ps.fonttype"]=42

In [ ]:
sc.set_figure_params(figsize=(4,4))
torch.set_float32_matmul_precision("high")

In [ ]:
project_ID="pool_Myeloid"
mtx="transcriptome"
sc.settings.figdir=f'{mtx}/{project_ID}_figures'
sc.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
scv.settings.figdir=f'{mtx}/{project_ID}_figures'
scv.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
if "Myeloid" not in exported_major_groups:
    raise ValueError("Myeloid: no cells exported from current broad annotation; check anno_clusters.")
adata=sc.read(f"{mtx}/{project_ID}_raw.h5ad")

In [ ]:
#store a h5ad file for further faster loadinig

#we need to make a directory to store the files in
try:
    directoryName=mtx+"/"+project_ID+"_figures"
    os.mkdir(directoryName)
except:
    pass

#we need to make a directory to store the files in

try:
    directoryName=mtx+"/"+project_ID+"_DEG"
    os.mkdir(directoryName)
except:
    pass


## Preprocess the Data

now we will take the files from the previous section where we determined spliced/unspliced RNA counts and use them to create RNA velocity trajectory as well as investigate biological alternation in each cell

In [ ]:
#plot scatter plots of dataset QC
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt',color="anno_clusters")
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_Rp',color="anno_clusters")
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts',color="anno_clusters")

In [ ]:
#filter out outliers
adata = adata[adata.obs.n_genes_by_counts < 7000, :] #we want the cell to contain less than 5000 total gene detected in order to avoid doublets
adata = adata[adata.obs.total_counts < 40000, :]
adata = adata[adata.obs.n_genes_by_counts > 400, :] #we want the cell to contain more than 200 total gene detected in order to avoid bad cells
adata = adata[adata.obs.total_counts > 800, :]# we want the cell to contain more than 400 (200*2) total counts (UMI) detected in order to avoid bad cells
adata = adata[adata.obs.pct_counts_mt < 10, :]# we want the cell to contain less than 55 percent mitochondrial genes in order to avoid bad cells

adata

**Doublets section**

This section should only be used if the dataframe contains doublets.


In [ ]:
#calculate doublets but we don't do this for SUM149 datasets
scrub = scr.Scrublet(
    adata.X, expected_doublet_rate=0.06,
    n_neighbors=max(5, int(round(0.5 * np.sqrt(adata.n_obs))))
)

In [ ]:
adata.obs['doublet_scores'], adata.obs['predicted_doublets'] = scrub.scrub_doublets(min_counts=5, 
                                                          min_cells=max(3, int(round(0.001 * adata.n_obs))), 
                                                          min_gene_variability_pctl=95, 
                                                          n_prin_comps=30)
scrub.plot_histogram()

In [ ]:
singlet_list = [cells for cells in adata.obs_names if not adata.obs['predicted_doublets'][cells]]
#adata = adata[singlet_list,:]

In [ ]:
adata=adata[adata.obs.doublet_scores<0.25]

In [ ]:
#plot scatter plots of dataset QC
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt',color="doublet_scores")

***

**Continued data processing**

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_filtered_raw.h5ad")

**Continued data processing**

In [ ]:
adata.layers["counts"] = adata.X.copy()  # preserve counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="Sample_id",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    categorical_covariate_keys=["Sample_id"],
    continuous_covariate_keys=['n_genes_by_counts','pct_counts_mt','total_counts', 'pct_counts_Rp'],
)

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model_dir = os.path.join(mtx, f"{project_ID}_figures", "scvi_model")
model.save(model_dir, overwrite=True)


In [ ]:
model = scvi.model.SCVI.load(model_dir, adata=adata)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
denoised = model.get_normalized_expression(adata, library_size=1e4)

In [ ]:
SCVI_NORMALIZED_KEY = "scvi_normalized"

adata.layers[SCVI_NORMALIZED_KEY] = model.get_normalized_expression(library_size=10e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)

In [ ]:
sc.pp.scale(adata,layer=SCVI_NORMALIZED_KEY )

In [ ]:
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_clusters"],
    frameon=False,
)
sc.pl.umap(
    adata,
    color=["Sample_id","treatment"],
    ncols=2,
    frameon=False,
)

In [ ]:
# neighbors were already computed using scVI
SCVI_CLUSTERS_KEY = "leiden_scVI"
sc.tl.leiden(adata, key_added=SCVI_CLUSTERS_KEY, resolution=0.8)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_scVI"],
    frameon=False,
)

In [ ]:
old_to_new={
    "0":"TAM.M2",
    "1":"TAM.Ccr2",
    "2":"Mo.DC",
    "3":"TAM.M2",
    "4":"TAM.Ccl",
    "5":"TAM.Fth1",
    "6":"mixed.cells",
    "7":"TAM.Ccr2",
    "8":"TAM.plac8",
    "9":"TAM.Ccr2",
    "10":"MDSC",
    "11":"TAM.prolif",
    "12":"DC.migrat",
    "13":"Gran.cells",
    "14":"pre.DC"
                
    
}

In [ ]:
adata.obs["anno_sub"]=(adata.obs["leiden_scVI"].map(old_to_new).astype("category"))

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_sub"],
    frameon=False,
)

In [ ]:
groupby="anno_sub"
sc.tl.dendrogram(adata,groupby=groupby)
markers ={
    "CD8":["Cd3g","Cd8a","Cd8b1"],
    "Myeloid.cells":["Itgam"],
    "MDSCs":["Ly6g","Ly6c1","Ly6c2","Lin28a","Cd33",],
    "Gran/Neu":["Fcgr3","Csf3r","Fut4","Cxcr4"],
    "Dendritic.cells":["Itgax","Clec9a","Clec10a","Siglech","Bst2","Cd300c","Lsp1"], 
    "Monocytes":["Cd14","Cd300a","F13a1","Hp","Vcan","Sparc"],
    "Macrophages":["Lyz2","Cd68","Csf1r","Adgre1",],
    "TREM":["Trem1","Trem2"],
    "M1_TAM":["Cd86","Cx3cr1","Cxcl16",],
    "M2_TAM":["Mrc1","C1qa","C1qb","C1qc","Siglec1","C5ar1"],
    "Microglia":["Tmem119","P2ry12"],
    "Immune_checkpoint":["Cd274","Cd80"],
    "MHC-II":['H2-K1','H2-Aa','H2-T23','H2-M5', 'H2-M2','H2-Oa','H2-DMb2','H2-Ob' ,'H2-Ab1', 'H2-Eb2', 'H2-D1','H2-Q7',]
}


sc.pl.dotplot(adata, markers,standard_scale="var",cmap="RdYlBu_r",figsize=(16,3),
                     mean_only_expressed=True, expression_cutoff=0,
                     groupby=groupby, dendrogram=True,save=f"{project_ID}_markers_{groupby}.pdf")

# DEG

In [ ]:
sc.tl.rank_genes_groups(adata, "anno_sub", method='wilcoxon', min_log_fold_change=2, 
                        n_genes=4000,#layer="scvi_normalized",
                        use_raw=True) #when we set layer="spliced", we can extract the DEGs of each cluster according to only normalized and highly virable spliced RNA matrix. When we don't set the layer, we calculate DEGs according to scaled spliced RNA matrix. When we set use_raw="True", we will get DEGs according to raw spliced matrix even without filtering highly virable genes or outliers.
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, rankby_obs=True)

In [ ]:
#save top50 DEG into dataframe, when we calculate the DEGs according to scaled matrix, because negative numbers in the matrix, we will not be able to get log fold chages, so that we can only summarize pvals_adj in to the dataframe
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
result_df=pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names' ,'pvals_adj',"logfoldchanges"]}).head(50)

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi.h5ad")

## ID8 tumor-cell subclustering


In [ ]:
%matplotlib inline

In [ ]:
scvi.settings.seed = 0

In [ ]:
sc.set_figure_params(figsize=(4,4))
torch.set_float32_matmul_precision("high")

In [ ]:
project_ID="pool_ID8"
mtx="transcriptome"
sc.settings.figdir=f'{mtx}/{project_ID}_figures'
sc.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
scv.settings.figdir=f'{mtx}/{project_ID}_figures'
scv.settings.figdir=f'{mtx}/{project_ID}_figures'

In [ ]:
if "ID8" not in exported_major_groups:
    raise ValueError("ID8: no cells exported from current broad annotation; check anno_clusters.")
adata=sc.read(f"{mtx}/{project_ID}_raw.h5ad")

In [ ]:
#store a h5ad file for further faster loadinig

#we need to make a directory to store the files in
try:
    directoryName=mtx+"/"+project_ID+"_figures"
    os.mkdir(directoryName)
except:
    pass

#we need to make a directory to store the files in

try:
    directoryName=mtx+"/"+project_ID+"_DEG"
    os.mkdir(directoryName)
except:
    pass


## Preprocess the Data

now we will take the files from the previous section where we determined spliced/unspliced RNA counts and use them to create RNA velocity trajectory as well as investigate biological alternation in each cell

In [ ]:
#plot scatter plots of dataset QC
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt',color="Sample_id")
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_Rp',color="Sample_id")
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts',color="anno_clusters")

In [ ]:
#filter out outliers
adata = adata[adata.obs.n_genes_by_counts < 7000, :] #we want the cell to contain less than 5000 total gene detected in order to avoid doublets
adata = adata[adata.obs.total_counts < 40000, :]
adata = adata[adata.obs.n_genes_by_counts > 800, :] #we want the cell to contain more than 200 total gene detected in order to avoid bad cells
adata = adata[adata.obs.total_counts > 1000, :]# we want the cell to contain more than 400 (200*2) total counts (UMI) detected in order to avoid bad cells
adata = adata[adata.obs.pct_counts_mt < 10, :]# we want the cell to contain less than 55 percent mitochondrial genes in order to avoid bad cells

adata

**Doublets section**

This section should only be used if the dataframe contains doublets.


In [ ]:
#calculate doublets but we don't do this for SUM149 datasets
scrub = scr.Scrublet(
    adata.X, expected_doublet_rate=0.06,
    n_neighbors=max(5, int(round(0.5 * np.sqrt(adata.n_obs))))
)

In [ ]:
adata.obs['doublet_scores'], adata.obs['predicted_doublets'] = scrub.scrub_doublets(min_counts=5, 
                                                          min_cells=max(3, int(round(0.001 * adata.n_obs))), 
                                                          min_gene_variability_pctl=95, 
                                                          n_prin_comps=30)
scrub.plot_histogram()

In [ ]:
singlet_list = [cells for cells in adata.obs_names if not adata.obs['predicted_doublets'][cells]]
#adata = adata[singlet_list,:]

In [ ]:
adata=adata[adata.obs.doublet_scores<0.25]

In [ ]:
#plot scatter plots of dataset QC
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt',color="doublet_scores")

***

**Continued data processing**

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_filtered_raw.h5ad")

**Continued data processing**

In [ ]:
adata.layers["counts"] = adata.X.copy()  # preserve counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key="Sample_id",
)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    categorical_covariate_keys=["Sample_id"],
    continuous_covariate_keys=['n_genes_by_counts','pct_counts_mt','total_counts', 'pct_counts_Rp'],
)

In [ ]:
model = scvi.model.SCVI(adata)

In [ ]:
model.train()

In [ ]:
model_dir = os.path.join(mtx, f"{project_ID}_figures", "scvi_model")
model.save(model_dir, overwrite=True)


In [ ]:
model = scvi.model.SCVI.load(model_dir, adata=adata)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
latent = model.get_latent_representation()
adata.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
denoised = model.get_normalized_expression(adata, library_size=1e4)

In [ ]:
SCVI_NORMALIZED_KEY = "scvi_normalized"

adata.layers[SCVI_NORMALIZED_KEY] = model.get_normalized_expression(library_size=10e4)


In [ ]:
# use scVI latent space for UMAP generation
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)

In [ ]:
sc.pp.scale(adata,layer=SCVI_NORMALIZED_KEY )

In [ ]:
sc.tl.umap(adata, min_dist=0.3)

In [ ]:

sc.pl.umap(
    adata,
    color=["Sample_id","treatment"],
    ncols=2,
    frameon=False,
)

In [ ]:
# neighbors were already computed using scVI
SCVI_CLUSTERS_KEY = "leiden_scVI"
sc.tl.leiden(adata, key_added=SCVI_CLUSTERS_KEY, resolution=0.52, use_weights=True)


In [ ]:
old_to_new={
    "0":"ID8.EMT",
    "1":"ID8.EMT.IFNg",
    "2":"ID8.prolif",
    "3":"ID8.ESR",
    "4":"ID8.adipo",
    "5":"ID8.IFNg",
    "6":"ID8.prolif.apop",
   # "7":"Malign_7",
   # "8":"Malign_8",
   # "9":"Malign_9",
   # "10":"Malign_10",
}

In [ ]:
adata.obs["anno_sub"]=(adata.obs["leiden_scVI"].map(old_to_new).astype("category"))

In [ ]:
sc.pl.umap(
    adata,
    color=["anno_sub"],
    frameon=False,
)

# DEG

In [ ]:
sc.tl.rank_genes_groups(adata, "anno_sub", method='wilcoxon', min_log_fold_change=2, 
                        n_genes=4000,#layer="scvi_normalized",
                        use_raw=True) #when we set layer="spliced", we can extract the DEGs of each cluster according to only normalized and highly virable spliced RNA matrix. When we don't set the layer, we calculate DEGs according to scaled spliced RNA matrix. When we set use_raw="True", we will get DEGs according to raw spliced matrix even without filtering highly virable genes or outliers.
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, rankby_obs=True)

In [ ]:
#save top50 DEG into dataframe, when we calculate the DEGs according to scaled matrix, because negative numbers in the matrix, we will not be able to get log fold chages, so that we can only summarize pvals_adj in to the dataframe
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
result_df=pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names' ,'pvals_adj',"logfoldchanges"]}).head(50)

In [ ]:
adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi2.h5ad")

## 其他大群：独立读取、聚类、marker 与 DEG

对 Epi、Fibro、Endo、NK、Unknown 分别使用自己的 raw 文件进行分析。只处理本轮实际导出的大群；没有细胞时跳过，避免读取旧文件。原大类映射没有 Fibro 标签，因此需先根据大类 marker 确认 Fibro 所属簇并更新前面的大类映射，才能得到 Fibro 结果。

原 CAF/Endo 片段没有可对应的新聚类命名表，已移除旧映射。以下 `anno_sub` 只表示“大群 + 聚类编号”，不代表已确认的生物学亚型。scVI 与 QC 沿用现有流程的做法，聚类分辨率暂用总群的 0.4；正式亚型命名需检查 marker。


In [ ]:
# Each iteration starts from that group's exported raw counts, never the previous adata.
additional_groups = ["Epi", "Fibro", "Endo", "NK", "Unknown"]
for group_name in additional_groups:
    if group_name not in exported_major_groups:
        print(f"{group_name}: no cells exported in this run; skipped subgroup analysis.")
        continue

    project_ID = f"pool_{group_name}"
    sc.settings.figdir = f"{mtx}/{project_ID}_figures"
    scv.settings.figdir = sc.settings.figdir
    os.makedirs(f"{mtx}/{project_ID}_figures", exist_ok=True)
    os.makedirs(f"{mtx}/{project_ID}_DEG", exist_ok=True)
    adata = sc.read(f"{mtx}/{project_ID}_raw.h5ad")
    # The parent population was already filtered; avoid transferring ID8-specific QC to other groups.
    if adata.n_obs < 10:
        print(f"{group_name}: only {adata.n_obs} cells; skipped subgroup analysis.")
        continue

    adata.layers["counts"] = adata.X.copy()
    # Keep the normalized/log1p raw reference inherited from the pooled analysis.
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(
        adata, n_top_genes=min(3000, adata.n_vars), subset=True,
        layer="counts", flavor="seurat_v3", batch_key="Sample_id",
    )
    scvi.settings.seed = 0
    scvi.model.SCVI.setup_anndata(
        adata, layer="counts", categorical_covariate_keys=["Sample_id"],
        continuous_covariate_keys=["n_genes_by_counts", "pct_counts_mt", "total_counts", "pct_counts_Rp"],
    )
    model = scvi.model.SCVI(adata)
    model.train()
    model.save(f"{mtx}/{project_ID}_figures/scvi_model", overwrite=True)
    adata.obsm["X_scVI"] = model.get_latent_representation()
    adata.layers["scvi_normalized"] = model.get_normalized_expression(library_size=1e4)
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=min(15, adata.n_obs - 1))
    sc.tl.umap(adata, min_dist=0.3)
    sc.tl.leiden(adata, key_added="leiden_scVI", resolution=0.4)
    adata.obs["anno_sub"] = (
        group_name + ".cluster_" + adata.obs["leiden_scVI"].astype(str)
    ).astype("category")
    sc.pl.umap(adata, color=["anno_sub", "Sample_id"], frameon=False,
               save=f"_{project_ID}_subgroups.pdf")

    # Discover subgroup markers without reusing historical CAF/Endo cluster labels.
    cluster_sizes = adata.obs["anno_sub"].value_counts()
    if (cluster_sizes > 0).sum() > 1 and cluster_sizes[cluster_sizes > 0].min() > 1:
        sc.tl.rank_genes_groups(adata, "anno_sub", method="wilcoxon", use_raw=True)
        sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False,
                               save=f"_{project_ID}_markers.pdf")
        sc.get.rank_genes_groups_df(adata, group=None).to_csv(
            f"{mtx}/{project_ID}_DEG/anno_sub_DEG.csv", index=False,
        )
    else:
        print(f"{group_name}: insufficient clusters or singleton cluster; skipped DEG.")
    adata.write(f"{mtx}/{project_ID}_figures/{project_ID}_anno_scvi.h5ad")
